In [1]:
!nvidia-smi

Fri Sep 25 07:13:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install ortools
!pip install pycuda

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 29.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 w

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 11.7 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp313-cp313-linux_x86_64.whl size=5280313 sha256=46d770e64bb2a768b21fa24431a86ce339221d87afc329f96b2c6f440f16de26
  Stored in directory: /root/.cache/pip/wheels/ce/26/46/c519675fcb0e5e17bab8e85b6676528c40d12d794182340e85
Successfully built pycuda


In [ ]:
from ortools.algorithms.python import knapsack_solver
import pycuda.autoinit
import pycuda.driver as cuda
import numpy as np
from pycuda.compiler import SourceModule
import time
import platform

In [4]:
# Colab starts with an empty filesystem, so pull the project in to get problems.py.
# Re-run this cell after a runtime restart: the clone survives, sys.path does not.
import os
import sys

REPO_URL = "https://github.com/andrewrowell/subset-sum-gpu.git"
REPO_DIR = "/content/subset-sum-gpu"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

Cloning into '/content/subset-sum-gpu'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 138 (delta 61), reused 101 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 233.82 KiB | 1.24 MiB/s, done.
Resolving deltas: 100% (61/61), done.


In [ ]:
NUMBER_OF_TRIALS = 100

import problems

# Every implementation in this project solves the identical set of problems by
# reading them from problems.py.
problem_set = problems.generate()

num_problems = problem_set.num_problems
num_items_per_problem = problem_set.num_items
max_capacity = problem_set.max_capacity
capacities = problem_set.capacities

items = problem_set.items

print(f"{num_problems} problems, {num_items_per_problem} items each, capacity at most {max_capacity}")
for i in range(3):
    row = ", ".join(str(item) for item in problem_set.items[i])
    print(f"Problem {i + 1}: capacity {problem_set.capacities[i]}, items [{row}]")

In [ ]:
# How much the answers discriminate: a problem whose items can reach its capacity
# exactly has an answer anything could guess, one that falls short does not.
problem_set.analyze()

In [ ]:
# OR-Tools Section

print(platform.node())

# Function to solve a single subset sum problem using OR-Tools
def solve_subset_sum(items, capacity, problem_idx, totals, chosen):
    solver = knapsack_solver.KnapsackSolver(
        knapsack_solver.KNAPSACK_DYNAMIC_PROGRAMMING_SOLVER, 'SubsetSumExample')

    # Subset sum is knapsack with an item's value equal to its weight, so
    # OR-Tools is given the same array for both.
    solver.init(items, [items], [capacity])

    totals[problem_idx] = solver.solve()
    for i in range(len(items)):
        chosen[problem_idx, i] = solver.best_solution_contains(i)

# Storage for the totals and for which items made up each one
ortools_totals = np.zeros(num_problems, dtype=np.int32)
ortools_chosen = np.zeros((num_problems, num_items_per_problem), dtype=np.uint8)

execution_times = []
for _ in range(NUMBER_OF_TRIALS):
    # Measure execution time
    start_time = time.time()

    for i in range(num_problems):
        solve_subset_sum(items[i], capacities[i], i, ortools_totals, ortools_chosen)

    # Print execution time
    end_time = time.time()
    execution_times.append(end_time - start_time)

print(f"Average CPU execution time: {(sum(execution_times) / NUMBER_OF_TRIALS):.6f} seconds")

# Each reported total has to be the sum of the items it names, and the best available
print(f"{problems.check_solutions(problem_set, ortools_totals, ortools_chosen)} bad solutions")

# Print the final results
#for i in range(num_problems):
for i in range(10):
    print(f"Problem {i + 1}: total {ortools_totals[i]} "
          f"from items {np.flatnonzero(ortools_chosen[i]).tolist()}")

In [ ]:
# Bitset DP Section
#
# The same recurrence the GPU kernels run, but with the reachable totals packed one
# per bit of a single integer instead of one per byte of a table, so a shift and an
# OR advance the whole table by one item. problems.best_subset keeps the reachable
# set after each item so it can walk backwards and recover the items themselves.
print(platform.node())

bitset_totals = np.zeros(num_problems, dtype=np.int32)
bitset_chosen = np.zeros((num_problems, num_items_per_problem), dtype=np.uint8)

execution_times = []
for _ in range(NUMBER_OF_TRIALS):
    # Measure execution time
    start_time = time.time()

    for i in range(num_problems):
        total, picked = problems.best_subset(items[i], int(capacities[i]))
        bitset_totals[i] = total
        bitset_chosen[i] = 0
        bitset_chosen[i, picked] = 1

    # Print execution time
    end_time = time.time()
    execution_times.append(end_time - start_time)

print(f"Average bitset DP execution time: {(sum(execution_times) / NUMBER_OF_TRIALS):.6f} seconds")

# Each reported total has to be the sum of the items it names, and the best available
print(f"{problems.check_solutions(problem_set, bitset_totals, bitset_chosen)} bad solutions")

# Print the results
#for i in range(num_problems):
for i in range(10):
    print(f"Problem {i + 1}: total {bitset_totals[i]} "
          f"from items {np.flatnonzero(bitset_chosen[i]).tolist()}")

In [ ]:
# CUDA kernel to solve multiple subset sum problems. One block solves one problem;
# its threads split the table between them.
#
# The reachable totals are packed one per bit, exactly as problems.best_subset packs
# them into a Python integer, so a shift and an OR advance the whole table by one item
# instead of touching one byte per total. The set after each item is kept, which is
# what lets the kernel walk backwards afterwards and recover the items themselves.
kernel_code = """
__global__ void subset_sum(const int *items, const int *capacities, int *max_values,
                           unsigned char *chosen, int num_items, int max_capacity) {
    int words = (max_capacity + 32) / 32;
    extern __shared__ unsigned int history[];

    // history + i * words is the set reachable using only the first i items.
    // Before any item, the only reachable total is zero.
    for (int j = threadIdx.x; j < words; j += blockDim.x) {
        history[j] = (j == 0) ? 1u : 0u;
    }
    __syncthreads();

    const int *problem_items = items + blockIdx.x * num_items;
    for (int i = 0; i < num_items; i++) {
        int item = problem_items[i];
        int word_shift = item >> 5;   // whole words the bits move up by
        int bit_shift = item & 31;    // and the leftover bits within a word
        unsigned int *current = history + i * words;
        unsigned int *next = current + words;

        for (int j = threadIdx.x; j < words; j += blockDim.x) {
            unsigned int shifted = 0;
            int high = j - word_shift;
            if (high >= 0) {
                shifted = current[high] << bit_shift;
                // A shift that is not a whole number of words also pulls in the top
                // bits of the word below. Shifting a 32 bit word by 32 is undefined,
                // so a bit_shift of zero has to skip that part.
                if (bit_shift != 0 && high >= 1) {
                    shifted |= current[high - 1] >> (32 - bit_shift);
                }
            }
            next[j] = current[j] | shifted;
        }
        __syncthreads();
    }

    // One thread reads off the answer and walks the history back to find the items
    if (threadIdx.x == 0) {
        unsigned int *reachable = history + num_items * words;

        int w = capacities[blockIdx.x];
        while (((reachable[w >> 5] >> (w & 31)) & 1u) == 0u) {
            w--;
        }
        max_values[blockIdx.x] = w;

        // If the total was already reachable without item i then item i was not
        // needed; otherwise it was, so subtract it and carry on down.
        unsigned char *problem_chosen = chosen + blockIdx.x * num_items;
        int remaining = w;
        for (int i = num_items - 1; i >= 0; i--) {
            unsigned int *without = history + i * words;
            if (((without[remaining >> 5] >> (remaining & 31)) & 1u) != 0u) {
                problem_chosen[i] = 0;
            } else {
                problem_chosen[i] = 1;
                remaining -= problem_items[i];
            }
        }
    }
}
"""

# Compile the kernel code
mod = SourceModule(kernel_code)
subset_sum = mod.get_function("subset_sum")

# Allocate and fill the device buffers once, outside the timed loop. Unlike Apple
# silicon, this really is a transfer across PCIe to separate device memory, so it is
# worth doing once rather than on every trial.
items_gpu = cuda.mem_alloc(items.nbytes)
capacities_gpu = cuda.mem_alloc(capacities.nbytes)
max_values_gpu = cuda.mem_alloc(num_problems * 4)
chosen_gpu = cuda.mem_alloc(num_problems * num_items_per_problem)
cuda.memcpy_htod(items_gpu, items)
cuda.memcpy_htod(capacities_gpu, capacities)

cuda_totals = np.zeros(num_problems, dtype=np.int32)
cuda_chosen = np.zeros((num_problems, num_items_per_problem), dtype=np.uint8)

# One snapshot of the packed table per item, plus the starting one
words = (max_capacity + 32) // 32
shared_memory_size = (num_items_per_problem + 1) * words * 4

# One thread per word of the table measured fastest on the M4, where the barrier after
# each item gets more expensive the more threads have to reach it. This has not been
# swept on a T4, where the best value may differ.
threads_per_block = min(words, 1024)

execution_times = []
for _ in range(NUMBER_OF_TRIALS):
    # Measure execution time
    start_time = time.time()

    # Launch the kernel
    subset_sum(items_gpu, capacities_gpu, max_values_gpu, chosen_gpu,
               np.int32(num_items_per_problem), np.int32(max_capacity),
               block=(threads_per_block, 1, 1), grid=(num_problems, 1),
               shared=shared_memory_size)

    # Copy the results back to the CPU; this synchronizes with the kernel. The Metal
    # notebook has no equivalent step: Apple silicon shares one pool of memory between
    # the CPU and GPU, so numpy can read the kernel's output buffers in place, while
    # here the results have to come back across PCIe from the T4's own memory.
    cuda.memcpy_dtoh(cuda_totals, max_values_gpu)
    cuda.memcpy_dtoh(cuda_chosen, chosen_gpu)

    # Print execution time
    end_time = time.time()
    execution_times.append(end_time - start_time)

print(f"Average PyCUDA execution time: {(sum(execution_times) / NUMBER_OF_TRIALS):.6f} seconds")

# Each reported total has to be the sum of the items it names, and the best available
print(f"{problems.check_solutions(problem_set, cuda_totals, cuda_chosen)} bad solutions")

# Print the results
#for i in range(num_problems):
for i in range(10):
    print(f"Problem {i + 1}: total {cuda_totals[i]} "
          f"from items {np.flatnonzero(cuda_chosen[i]).tolist()}")